In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder\
            .appName("Bucketing & Segmentation")\
            .getOrCreate()

In [0]:
customers_data = [
(101,"Aarav","Hyderabad"),
(102,"Bhavya","Bangalore"),
(103,"Charan","Chennai"),
(104,"Divya","Mumbai"),
(105,"Esha","Pune"),
(106,"Farhan","Delhi"),
(107,"Gopal","Hyderabad"),
(108,"Harsha","Chennai"),
(109,"Ishita","Bangalore"),
(110,"John","Mumbai"),
(111,"Keerthi","Pune"),
(112,"Lokesh","Delhi"),
(113,"Meena","Hyderabad"),
(114,"Nikhil","Chennai"),
(115,"Pooja","Bangalore")
]

customer_columns = [
    "customer_id",
    "customer_name",
    "city"
]

customers_df = spark.createDataFrame(customers_data, customer_columns)

customers_df.show()

orders_data = [
(1001,101,"Laptop",52000),
(1002,101,"Mouse",1200),
(1003,102,"Mobile",28000),
(1004,103,"Headphones",2500),
(1005,103,"Keyboard",1800),
(1006,104,"Monitor",14000),
(1007,104,"Printer",11000),
(1008,105,"Chair",6000),
(1009,106,"Desk",8500),
(1010,106,"Laptop",51000),
(1011,107,"Mouse",1000),
(1012,107,"Keyboard",1700),
(1013,108,"Mobile",30000),
(1014,109,"Laptop",54000),
(1015,109,"Monitor",13500),
(1016,110,"Printer",12000),
(1017,110,"Chair",5500),
(1018,111,"Desk",9000),
(1019,111,"Mouse",1300),
(1020,112,"Laptop",50000),
(1021,112,"Keyboard",2000),
(1022,114,"Headphones",3000),
(1023,114,"Monitor",15000)
]

order_columns = [
    "order_id",
    "customer_id",
    "product",
    "order_amount"
]

orders_df = spark.createDataFrame(orders_data, order_columns)

orders_df.show()

+-----------+-------------+---------+
|customer_id|customer_name|     city|
+-----------+-------------+---------+
|        101|        Aarav|Hyderabad|
|        102|       Bhavya|Bangalore|
|        103|       Charan|  Chennai|
|        104|        Divya|   Mumbai|
|        105|         Esha|     Pune|
|        106|       Farhan|    Delhi|
|        107|        Gopal|Hyderabad|
|        108|       Harsha|  Chennai|
|        109|       Ishita|Bangalore|
|        110|         John|   Mumbai|
|        111|      Keerthi|     Pune|
|        112|       Lokesh|    Delhi|
|        113|        Meena|Hyderabad|
|        114|       Nikhil|  Chennai|
|        115|        Pooja|Bangalore|
+-----------+-------------+---------+

+--------+-----------+----------+------------+
|order_id|customer_id|   product|order_amount|
+--------+-----------+----------+------------+
|    1001|        101|    Laptop|       52000|
|    1002|        101|     Mouse|        1200|
|    1003|        102|    Mobile|       28

##Customer Spending Summary

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import *

In [0]:
customer_summary_df = customers_df.join(orders_df, on="customer_id", how="left")\
    .groupBy("customer_id", "customer_name", "city")\
    .agg(
        F.sum("order_amount").alias("total_spend"),
        F.count("order_id").alias("order_count")).fillna(0)

customer_summary_df.show()

+-----------+-------------+---------+-----------+-----------+
|customer_id|customer_name|     city|total_spend|order_count|
+-----------+-------------+---------+-----------+-----------+
|        101|        Aarav|Hyderabad|      53200|          2|
|        102|       Bhavya|Bangalore|      28000|          1|
|        103|       Charan|  Chennai|       4300|          2|
|        105|         Esha|     Pune|       6000|          1|
|        104|        Divya|   Mumbai|      25000|          2|
|        106|       Farhan|    Delhi|      59500|          2|
|        107|        Gopal|Hyderabad|       2700|          2|
|        109|       Ishita|Bangalore|      67500|          2|
|        108|       Harsha|  Chennai|      30000|          1|
|        110|         John|   Mumbai|      17500|          2|
|        111|      Keerthi|     Pune|      10300|          2|
|        113|        Meena|Hyderabad|          0|          0|
|        112|       Lokesh|    Delhi|      52000|          2|
|       

#Tasks

###1.  Gold/Silver/Bronze segmentation using conditional logic

In [0]:
segment_df = customer_summary_df.withColumn("segment",
                    when(F.col("total_spend")>50000, "Gold")\
                    .when(F.col("total_spend")>=20000,"Silver")\
                    .otherwise("Bronze"))
segment_df.show()

+-----------+-------------+---------+-----------+-----------+-------+
|customer_id|customer_name|     city|total_spend|order_count|segment|
+-----------+-------------+---------+-----------+-----------+-------+
|        101|        Aarav|Hyderabad|      53200|          2|   Gold|
|        102|       Bhavya|Bangalore|      28000|          1| Silver|
|        103|       Charan|  Chennai|       4300|          2| Bronze|
|        105|         Esha|     Pune|       6000|          1| Bronze|
|        104|        Divya|   Mumbai|      25000|          2| Silver|
|        106|       Farhan|    Delhi|      59500|          2|   Gold|
|        107|        Gopal|Hyderabad|       2700|          2| Bronze|
|        109|       Ishita|Bangalore|      67500|          2|   Gold|
|        108|       Harsha|  Chennai|      30000|          1| Silver|
|        110|         John|   Mumbai|      17500|          2| Bronze|
|        111|      Keerthi|     Pune|      10300|          2| Bronze|
|        113|       

###2. Group data by segment and count customers


In [0]:
grouped_df = segment_df.groupBy("segment")\
                .agg(count("segment").alias("customer_count"))
                
grouped_df.show()

+-------+--------------+
|segment|customer_count|
+-------+--------------+
|   Gold|             4|
| Silver|             3|
| Bronze|             8|
+-------+--------------+



In [0]:
segment_df.createOrReplaceTempView("customers")

result_df = spark.sql("""
    SELECT 
        customer_id, 
        customer_name, 
        city, 
        total_spend, 
        CASE 
            WHEN total_spend > 50000 THEN 'Gold' 
            WHEN total_spend >= 20000 THEN 'Silver' 
            ELSE 'Bronze' 
        END AS segment 
    FROM customers
""")

result_df.show()


+-----------+-------------+---------+-----------+-------+
|customer_id|customer_name|     city|total_spend|segment|
+-----------+-------------+---------+-----------+-------+
|        101|        Aarav|Hyderabad|      53200|   Gold|
|        102|       Bhavya|Bangalore|      28000| Silver|
|        103|       Charan|  Chennai|       4300| Bronze|
|        105|         Esha|     Pune|       6000| Bronze|
|        104|        Divya|   Mumbai|      25000| Silver|
|        106|       Farhan|    Delhi|      59500|   Gold|
|        107|        Gopal|Hyderabad|       2700| Bronze|
|        109|       Ishita|Bangalore|      67500|   Gold|
|        108|       Harsha|  Chennai|      30000| Silver|
|        110|         John|   Mumbai|      17500| Bronze|
|        111|      Keerthi|     Pune|      10300| Bronze|
|        113|        Meena|Hyderabad|          0| Bronze|
|        112|       Lokesh|    Delhi|      52000|   Gold|
|        115|        Pooja|Bangalore|          0| Bronze|
|        114| 

### Segmentation using Bucketizer (MLlib)

In [0]:
from pyspark.ml.feature import Bucketizer


splits = [-float("inf"),20000,50000,float("inf")]

bucketizer = Bucketizer(
    splits=splits,
    inputCol="total_spend",
    outputCol="bucket"
)

bucket_df = bucketizer.transform(customer_summary_df)

bucket_df.show()

+-----------+-------------+---------+-----------+-----------+------+
|customer_id|customer_name|     city|total_spend|order_count|bucket|
+-----------+-------------+---------+-----------+-----------+------+
|        101|        Aarav|Hyderabad|      53200|          2|   2.0|
|        102|       Bhavya|Bangalore|      28000|          1|   1.0|
|        103|       Charan|  Chennai|       4300|          2|   0.0|
|        105|         Esha|     Pune|       6000|          1|   0.0|
|        104|        Divya|   Mumbai|      25000|          2|   1.0|
|        106|       Farhan|    Delhi|      59500|          2|   2.0|
|        107|        Gopal|Hyderabad|       2700|          2|   0.0|
|        109|       Ishita|Bangalore|      67500|          2|   2.0|
|        108|       Harsha|  Chennai|      30000|          1|   1.0|
|        110|         John|   Mumbai|      17500|          2|   0.0|
|        111|      Keerthi|     Pune|      10300|          2|   0.0|
|        113|        Meena|Hyderab

## Quantile-based Segmentation


In [0]:
quantiles = customer_summary_df.approxQuantile(
    "total_spend",
    [0.33,0.66],
    0
)

print(quantiles)

[6000.0, 28000.0]


In [0]:
q1,q2 = quantiles

quantile_df = customer_summary_df.withColumn(
    "quantile_segment",
    when(col("total_spend") <= q1,"Low")
    .when(col("total_spend") <= q2,"Medium")
    .otherwise("High")
)

display(quantile_df)

customer_id,customer_name,city,total_spend,order_count,quantile_segment
101,Aarav,Hyderabad,53200,2,High
102,Bhavya,Bangalore,28000,1,Medium
103,Charan,Chennai,4300,2,Low
105,Esha,Pune,6000,1,Low
104,Divya,Mumbai,25000,2,Medium
106,Farhan,Delhi,59500,2,High
107,Gopal,Hyderabad,2700,2,Low
109,Ishita,Bangalore,67500,2,High
108,Harsha,Chennai,30000,1,High
110,John,Mumbai,17500,2,Medium


##Window-based Ranking

In [0]:

from pyspark.sql.window import Window
from pyspark.sql.functions import percent_rank

window = Window.orderBy("total_spend")

rank_df = customer_summary_df.withColumn(
    "percent_rank",
    percent_rank().over(window)
)

display(rank_df)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


customer_id,customer_name,city,total_spend,order_count,percent_rank
113,Meena,Hyderabad,0,0,0.0
115,Pooja,Bangalore,0,0,0.0
107,Gopal,Hyderabad,2700,2,0.14285714285714285
103,Charan,Chennai,4300,2,0.21428571428571427
105,Esha,Pune,6000,1,0.2857142857142857
111,Keerthi,Pune,10300,2,0.35714285714285715
110,John,Mumbai,17500,2,0.42857142857142855
114,Nikhil,Chennai,18000,2,0.5
104,Divya,Mumbai,25000,2,0.5714285714285714
102,Bhavya,Bangalore,28000,1,0.6428571428571429


##Comparison between Methods

In [0]:
comparison_df = segment_df.join(
    bucket_df.select("customer_id","bucket"),
    "customer_id"
).join(
    quantile_df.select("customer_id","quantile_segment"),
    "customer_id"
).join(
    rank_df.select("customer_id","percent_rank"),
    "customer_id"
)

display(comparison_df)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


customer_id,customer_name,city,total_spend,order_count,segment,bucket,quantile_segment,percent_rank
101,Aarav,Hyderabad,53200,2,Gold,2.0,High,0.8571428571428571
102,Bhavya,Bangalore,28000,1,Silver,1.0,Medium,0.6428571428571429
103,Charan,Chennai,4300,2,Bronze,0.0,Low,0.21428571428571427
105,Esha,Pune,6000,1,Bronze,0.0,Low,0.2857142857142857
104,Divya,Mumbai,25000,2,Silver,1.0,Medium,0.5714285714285714
106,Farhan,Delhi,59500,2,Gold,2.0,High,0.9285714285714286
107,Gopal,Hyderabad,2700,2,Bronze,0.0,Low,0.14285714285714285
109,Ishita,Bangalore,67500,2,Gold,2.0,High,1.0
108,Harsha,Chennai,30000,1,Silver,1.0,High,0.7142857142857143
110,John,Mumbai,17500,2,Bronze,0.0,Medium,0.42857142857142855
